In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# TCGA-LIHC Step15B
# Stability Validation
# Repeated CV + Bootstrap + Train-only Feature Selection
# ============================================================

from pathlib import Path
import os
import json
import hashlib
import shutil
from datetime import datetime

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. 项目路径
# ------------------------------------------------------------

ROOT = Path(_hcc_path(''))

# 原 Step15：只读，不修改
SOURCE_DIR = ROOT / "LIHC_RNA_Step15_Machine_Learning"

# 新 Step15B
PROJECT_DIR = ROOT / "LIHC_RNA_Step15_Machine_Learning_Stability_Validation"

INPUT_DIR = PROJECT_DIR / "Input"
RESULTS_DIR = PROJECT_DIR / "Results"
FIGURES_DIR = PROJECT_DIR / "Figures"
MODELS_DIR = PROJECT_DIR / "Models"
AUDIT_DIR = PROJECT_DIR / "Audit"
NOTEBOOK_DIR = PROJECT_DIR / "Notebook"

for folder in [
    PROJECT_DIR,
    INPUT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    AUDIT_DIR,
    NOTEBOOK_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("TCGA-LIHC Step15B")
print("Repeated CV + Bootstrap + Train-only Feature Selection")
print("=" * 70)
print(f"Source Step15 : {SOURCE_DIR}")
print(f"Step15B       : {PROJECT_DIR}")
print()
print("目录创建完成。")

In [ ]:
# ============================================================
# Step15B 参数锁定
# ============================================================

RANDOM_STATE = 20260921

# Repeated CV
N_SPLITS = 5
N_REPEATS = 5

# Bootstrap
N_BOOTSTRAP = 1000

# Elastic Net Cox
# Step15B 内部固定，不再重新搜索超参数
ELASTIC_NET_PENALIZER = 0.010696
ELASTIC_NET_L1_RATIO = 0.50

# Random Survival Forest
RSF_N_ESTIMATORS = 300
RSF_MIN_SAMPLES_SPLIT = 10
RSF_MIN_SAMPLES_LEAF = 5
RSF_MAX_FEATURES = "sqrt"
# One RSF is fitted at a time, so use all available cores for each fit.
RSF_N_JOBS = -1

# RSF permutation importance
RSF_PERMUTATION_REPEATS = 5
RSF_TOP_N = 10

# 稳定性定义：
# 不在 Step15B 强行锁定新的最终基因，
# 先完整输出 selection frequency。
STABILITY_FREQ_THRESHOLD = 0.60

# 是否运行“真正的 fold-specific train-only selection 模型”
# True = 完整运行
# False = 只做固定特征集 repeated CV + selection stability
RUN_DYNAMIC_SELECTION_PERFORMANCE = True

# 是否运行 RSF
RUN_RSF = True

VERSION = "Step15B_v2.0"

RUN_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CONFIG = {
    "version": VERSION,
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "n_repeats": N_REPEATS,
    "n_bootstrap": N_BOOTSTRAP,
    "elastic_net_penalizer": ELASTIC_NET_PENALIZER,
    "elastic_net_l1_ratio": ELASTIC_NET_L1_RATIO,
    "rsf_n_estimators": RSF_N_ESTIMATORS,
    "rsf_min_samples_split": RSF_MIN_SAMPLES_SPLIT,
    "rsf_min_samples_leaf": RSF_MIN_SAMPLES_LEAF,
    "rsf_max_features": RSF_MAX_FEATURES,
    "rsf_n_jobs": RSF_N_JOBS,
    "rsf_permutation_repeats": RSF_PERMUTATION_REPEATS,
    "rsf_top_n": RSF_TOP_N,
    "stability_freq_threshold": STABILITY_FREQ_THRESHOLD,
    "run_dynamic_selection_performance": RUN_DYNAMIC_SELECTION_PERFORMANCE,
    "run_rsf": RUN_RSF,
    "run_time": RUN_TIME,
}

with open(AUDIT_DIR / "STEP15B_RUN_CONFIG.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# Step15A -> Step15B 输入文件复制
# ============================================================

SOURCE_FILES = [
    "LIHC_Multilevel_Master_Feature_Table.csv",
    "LIHC_Survival_Train_Test_Split_70_30.csv",

    "Stable_Prognostic_Genes_10.csv",
    "Stable_Biological_Features.csv",

    "Gene_Definition_Master_Table.csv",
    "Feature_Definition_Master_Table.csv",
    "FEATURE_SELECTION_LINEAGE.csv",
    "MODEL_VERSION_LOG.csv",

    "LOCKED_FEATURE_DEFINITIONS_v1.0.json",
]

copied_records = []

for filename in SOURCE_FILES:

    src = SOURCE_DIR / "Input" / filename

    # Step15A 中有些文件位于 Results / Audit
    if not src.exists():
        candidates = [
            SOURCE_DIR / "Results" / filename,
            SOURCE_DIR / "Audit" / filename,
        ]

        for candidate in candidates:
            if candidate.exists():
                src = candidate
                break

    if not src.exists():
        raise FileNotFoundError(f"Step15A 找不到文件：{filename}")

    dst = INPUT_DIR / filename

    shutil.copy2(src, dst)

    copied_records.append({
        "File": filename,
        "Source": str(src),
        "Destination": str(dst),
        "Size_KB": round(dst.stat().st_size / 1024, 2),
    })

input_manifest = pd.DataFrame(copied_records)

input_manifest.to_csv(
    AUDIT_DIR / "STEP15B_INPUT_COPY_MANIFEST.csv",
    index=False,
    encoding="utf-8-sig"
)

display(input_manifest)

print("\nStep15A 锁定输入已复制到 Step15B/Input。")
print("原 Step15A 文件不会被修改。")

In [ ]:
# ============================================================
# 输入文件 SHA256 完整性审计
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


hash_records = []

for filename in SOURCE_FILES:

    src_candidates = [
        SOURCE_DIR / "Input" / filename,
        SOURCE_DIR / "Results" / filename,
        SOURCE_DIR / "Audit" / filename,
    ]

    src = next((p for p in src_candidates if p.exists()), None)

    dst = INPUT_DIR / filename

    if src is None:
        continue

    hash_records.append({
        "File": filename,
        "Source_SHA256": sha256_file(src),
        "Copied_SHA256": sha256_file(dst),
        "Match": sha256_file(src) == sha256_file(dst),
    })

hash_df = pd.DataFrame(hash_records)

hash_df.to_csv(
    AUDIT_DIR / "STEP15B_INPUT_SHA256_AUDIT.csv",
    index=False,
    encoding="utf-8-sig"
)

display(hash_df)

if not hash_df["Match"].all():
    raise RuntimeError("输入文件 SHA256 不一致，请停止分析。")

print("\nSHA256 完整性检查 PASS。")

In [ ]:
# ============================================================
# Step15B 环境
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

print("Python / sklearn / lifelines / scikit-survival 导入完成。")

In [ ]:
# ============================================================
# 读取 Step15A 锁定数据
# ============================================================

MASTER_FILE = INPUT_DIR / "LIHC_Multilevel_Master_Feature_Table.csv"
SPLIT_FILE = INPUT_DIR / "LIHC_Survival_Train_Test_Split_70_30.csv"

master = pd.read_csv(MASTER_FILE)
split_df = pd.read_csv(SPLIT_FILE)

print("Master:", master.shape)
print("Train/Test split:", split_df.shape)

display(master.head())
display(split_df.head())

In [ ]:
# ============================================================
# 识别 Sample ID
# ============================================================

def find_id_column(df):
    candidates = [
        "Sample_ID",
        "ID",
        "Patient_ID",
        "sample_id",
        "patient_id",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError(
        f"无法识别 ID 列。当前列名：{list(df.columns)}"
    )


master_id_col = find_id_column(master)
split_id_col = find_id_column(split_df)

print("Master ID:", master_id_col)
print("Split ID :", split_id_col)

In [ ]:
# ============================================================
# 识别 group
# ============================================================

group_candidates = ["group", "Group", "set", "Set", "dataset", "Dataset"]

split_group_col = None

for c in group_candidates:
    if c in split_df.columns:
        split_group_col = c
        break

if split_group_col is None:
    raise KeyError(
        f"找不到 Train/Test group 列：{list(split_df.columns)}"
    )

print("Group column:", split_group_col)

print(
    split_df[split_group_col]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# 建立 340 人 Step15B 分析队列
# ============================================================

analysis_ids = split_df[split_id_col].astype(str)

master["_Sample_ID_Internal"] = master[master_id_col].astype(str)

analysis_df = master[
    master["_Sample_ID_Internal"].isin(set(analysis_ids))
].copy()

analysis_df = analysis_df.drop_duplicates(
    subset="_Sample_ID_Internal"
).copy()

print("Step15B analysis cohort:", analysis_df.shape)

# 必须严格等于 340
if len(analysis_df) != len(split_df):
    raise RuntimeError(
        f"Step15A 340人队列匹配失败："
        f"master匹配={len(analysis_df)}, split={len(split_df)}"
    )

analysis_df.to_csv(
    INPUT_DIR / "LIHC_Step15B_Analysis_Cohort_340.csv",
    index=False,
    encoding="utf-8-sig"
)

print("340人分析队列锁定。")

In [ ]:
# ============================================================
# Outcome 审计
# ============================================================

TIME_COL = "OS_time"
EVENT_COL = "OS"

if TIME_COL not in analysis_df.columns:
    raise KeyError(f"缺少 {TIME_COL}")

if EVENT_COL not in analysis_df.columns:
    raise KeyError(f"缺少 {EVENT_COL}")

analysis_df[TIME_COL] = pd.to_numeric(
    analysis_df[TIME_COL],
    errors="coerce"
)

analysis_df[EVENT_COL] = pd.to_numeric(
    analysis_df[EVENT_COL],
    errors="coerce"
)

if analysis_df[[TIME_COL, EVENT_COL]].isna().any().any():
    raise RuntimeError("OS_time / OS 存在缺失。")

analysis_df = analysis_df[
    analysis_df[TIME_COL] > 0
].copy()

analysis_df[EVENT_COL] = analysis_df[EVENT_COL].astype(int)

print("N =", len(analysis_df))
print("Events =", analysis_df[EVENT_COL].sum())
print(
    "Event rate =",
    round(analysis_df[EVENT_COL].mean(), 4)
)

print(
    "OS time:",
    analysis_df[TIME_COL].min(),
    "~",
    analysis_df[TIME_COL].max()
)

In [ ]:
# ============================================================
# 锁定历史 Original Top10
# ============================================================

ORIGINAL_TOP10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P",
]

# ============================================================
# 锁定严格 WGCNA Hub18
# ============================================================

HUB18 = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9",
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A",
]

GENE_POOL = list(
    dict.fromkeys(ORIGINAL_TOP10 + HUB18)
)

print("Original Top10:", len(ORIGINAL_TOP10))
print("Hub18:", len(HUB18))
print("Gene pool:", len(GENE_POOL))

if len(GENE_POOL) != 28:
    raise RuntimeError("28-gene candidate pool 数量错误。")

missing_genes = [
    g for g in GENE_POOL
    if g not in analysis_df.columns
]

if missing_genes:
    raise KeyError(
        f"以下 gene pool 不在 Master Feature Table：{missing_genes}"
    )

print("28-gene pool PASS")

In [ ]:
# ============================================================
# Biological feature pool
# ============================================================

# Derive the locked clinical encodings from raw Step15A columns.
# Unknown categories (TX/NX/MX or missing) remain NaN and are imputed within
# each training fold by the model preprocessing functions.
analysis_df["Gender_binary"] = analysis_df["Gender"].astype(str).str.lower().map({"female": 0, "male": 1})
analysis_df["Stage_numeric"] = pd.to_numeric(analysis_df["Stage"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
analysis_df["T_numeric"] = pd.to_numeric(analysis_df["T"].astype(str).str.extract(r"^T(\d+)")[0], errors="coerce")
analysis_df["N_numeric"] = pd.to_numeric(analysis_df["N"].astype(str).str.extract(r"^N(\d+)")[0], errors="coerce")
analysis_df["M_numeric"] = pd.to_numeric(analysis_df["M"].astype(str).str.extract(r"^M(\d+)")[0], errors="coerce")

clinical_encoding_audit = pd.DataFrame({
    "Feature": ["Gender_binary", "Stage_numeric", "T_numeric", "N_numeric", "M_numeric"],
    "Source": ["Gender", "Stage", "T", "N", "M"],
    "Rule": [
        "female=0; male=1",
        "Stage Roman numeral (I-IV) to 1-4; missing=NaN",
        "T category numeric component; TX/missing=NaN",
        "N category numeric component; NX/missing=NaN",
        "M category numeric component; MX/missing=NaN",
    ],
    "N_Missing": [
        int(analysis_df["Gender_binary"].isna().sum()),
        int(analysis_df["Stage_numeric"].isna().sum()),
        int(analysis_df["T_numeric"].isna().sum()),
        int(analysis_df["N_numeric"].isna().sum()),
        int(analysis_df["M_numeric"].isna().sum()),
    ],
})
clinical_encoding_audit.to_csv(AUDIT_DIR / "STEP15B_CLINICAL_ENCODING_AUDIT.csv", index=False, encoding="utf-8-sig")
display(clinical_encoding_audit)

CLINICAL_FEATURES = [
    "Age",
    "Gender_binary",
    "Stage_numeric",
    "T_numeric",
    "N_numeric",
    "M_numeric",
]

RISK_SCORE_FEATURE = ["RiskScore"]

OUTCOME_AND_ID_COLUMNS = {
    master_id_col,
    "_Sample_ID_Internal",
    "Patient_ID",
    "Sample_ID",
    "ID",
    "OS",
    "OS_time",
    "group",
    "Group",
    "label",
    "Label",
    "Gender",
    "Stage",
    "T",
    "N",
    "M",
    "A_Hub18_K2_Name",
    "D_Top30_each_K2_Name",
    "E_MM80_GS20_K2_Name",
}

excluded_features = set(
    GENE_POOL
    + CLINICAL_FEATURES
    + RISK_SCORE_FEATURE
) | OUTCOME_AND_ID_COLUMNS

BIO_POOL = [
    c for c in analysis_df.columns
    if c not in excluded_features
]

print("Biological feature pool:", len(BIO_POOL))

if len(BIO_POOL) != 86:
    print("\n当前识别出的 biological features：")
    print(BIO_POOL)

    raise RuntimeError(
        f"预期 biological feature = 86，"
        f"实际 = {len(BIO_POOL)}。"
        f"请不要继续运行，需要先检查 Feature Family。"
    )

print("86 biological features PASS")

In [ ]:
# ============================================================
# StableGenes10
# ============================================================

stable_gene_df = pd.read_csv(
    INPUT_DIR / "Stable_Prognostic_Genes_10.csv"
)

gene_name_col = "Gene"

if gene_name_col not in stable_gene_df.columns:
    gene_name_col = stable_gene_df.columns[0]

STABLE_GENES10 = (
    stable_gene_df[gene_name_col]
    .dropna()
    .astype(str)
    .tolist()
)

# ============================================================
# StableBio7
# ============================================================

stable_bio_df = pd.read_csv(
    INPUT_DIR / "Stable_Biological_Features.csv"
)

bio_name_col = "Feature"

if bio_name_col not in stable_bio_df.columns:
    bio_name_col = stable_bio_df.columns[0]

STABLE_BIO7 = (
    stable_bio_df[bio_name_col]
    .dropna()
    .astype(str)
    .tolist()
)

print("StableGenes10 =", len(STABLE_GENES10))
print(STABLE_GENES10)

print("\nStableBio7 =", len(STABLE_BIO7))
print(STABLE_BIO7)

if len(STABLE_GENES10) != 10:
    raise RuntimeError("StableGenes10 数量错误。")

if len(STABLE_BIO7) != 7:
    raise RuntimeError("StableBio7 数量错误。")

if not set(STABLE_GENES10).issubset(set(GENE_POOL)):
    raise RuntimeError("StableGenes10 不属于 28-gene pool。")

if not set(STABLE_BIO7).issubset(set(BIO_POOL)):
    raise RuntimeError("StableBio7 存在不属于 86 biological pool 的特征。")

print("\nStable feature definitions PASS")

In [ ]:
# ============================================================
# 固定 Feature Sets
# ============================================================

GENES10_BIO7 = list(
    dict.fromkeys(
        STABLE_GENES10 + STABLE_BIO7
    )
)

MODEL_FEATURE_SETS = {

    "Original_Top10": ORIGINAL_TOP10,

    "StableGenes10": STABLE_GENES10,

    "StableBio7": STABLE_BIO7,

    "Genes10_Bio7": GENES10_BIO7,

    "Clinical": CLINICAL_FEATURES,

    "RiskScore_Historical_Benchmark": [
        "RiskScore"
    ],

    "Clinical_RiskScore": [
        *CLINICAL_FEATURES,
        "RiskScore"
    ],

    "Clinical_StableGenes": [
        *CLINICAL_FEATURES,
        *STABLE_GENES10
    ],
}

for name, features in MODEL_FEATURE_SETS.items():

    missing = [
        f for f in features
        if f not in analysis_df.columns
    ]

    if missing:
        raise KeyError(
            f"{name} 缺少特征：{missing}"
        )

print("固定模型 Feature Sets：")

for name, features in MODEL_FEATURE_SETS.items():
    print(
        f"{name:<32} N={len(features)}"
    )

if len(GENES10_BIO7) != 17:
    raise RuntimeError("Genes10_Bio7 应为17个特征。")

In [ ]:
# ============================================================
# Repeated Stratified CV
# ============================================================

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

X_event = analysis_df[EVENT_COL].astype(int).values

CV_SPLITS = []

for split_index, (train_idx, valid_idx) in enumerate(
    cv.split(
        analysis_df,
        X_event
    )
):

    repeat_id = split_index // N_SPLITS + 1
    fold_id = split_index % N_SPLITS + 1

    CV_SPLITS.append({
        "repeat": repeat_id,
        "fold": fold_id,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
    })

print(
    f"Repeated CV: {N_REPEATS} repeats × {N_SPLITS} folds"
)

print(
    "Total outer folds =",
    len(CV_SPLITS)
)

In [ ]:
# ============================================================
# 保存每次 CV 的 Train / Validation 身份
# ============================================================

sample_ids = analysis_df[master_id_col].astype(str).values

cv_manifest = []

for split_info in CV_SPLITS:

    repeat_id = split_info["repeat"]
    fold_id = split_info["fold"]

    train_idx = split_info["train_idx"]
    valid_idx = split_info["valid_idx"]

    for idx in train_idx:

        cv_manifest.append({
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Sample_ID": sample_ids[idx],
            "Role": "Train",
        })

    for idx in valid_idx:

        cv_manifest.append({
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Sample_ID": sample_ids[idx],
            "Role": "Validation",
        })

cv_manifest_df = pd.DataFrame(cv_manifest)

cv_manifest_df.to_csv(
    RESULTS_DIR / "RepeatedCV_Split_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    RESULTS_DIR / "RepeatedCV_Split_Manifest.csv"
)

In [ ]:
# ============================================================
# Train-only Univariate Cox
# ============================================================

def train_only_univariate_cox(
    X_train,
    time_train,
    event_train,
    features,
):
    records = []

    for feature in features:

        x = pd.to_numeric(
            X_train[feature],
            errors="coerce"
        )

        tmp = pd.DataFrame({
            feature: x,
            "OS_time": time_train,
            "OS": event_train,
        }).dropna()

        if len(tmp) < 30:
            continue

        if tmp[feature].nunique() < 2:
            continue

        try:

            cph = CoxPHFitter()

            cph.fit(
                tmp,
                duration_col="OS_time",
                event_col="OS",
                show_progress=False
            )

            row = cph.summary.loc[feature]

            records.append({
                "Feature": feature,
                "HR": np.exp(row["coef"]),
                "Coef": row["coef"],
                "P": row["p"],
                "N": len(tmp),
            })

        except Exception:
            continue

    result = pd.DataFrame(records)

    if result.empty:
        return result

    result["Selected"] = result["P"] < 0.05

    return result.sort_values("P")

In [ ]:
# ============================================================
# Train-only ElasticNet Cox
# ============================================================

def train_only_elastic_net_selection(
    X_train,
    time_train,
    event_train,
    features,
):

    X = X_train[features].copy()

    # median imputation
    imputer = SimpleImputer(
        strategy="median"
    )

    X_imp = imputer.fit_transform(X)

    # remove zero variance
    variances = np.var(
        X_imp,
        axis=0
    )

    keep_mask = variances > 1e-12

    kept_features = [
        f for f, keep
        in zip(features, keep_mask)
        if keep
    ]

    if len(kept_features) == 0:
        return pd.DataFrame(
            columns=[
                "Feature",
                "Coefficient",
                "Selected"
            ]
        )

    X_imp = X_imp[:, keep_mask]

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X_imp
    )

    df_fit = pd.DataFrame(
        X_scaled,
        columns=kept_features
    )

    df_fit["OS_time"] = np.asarray(
        time_train,
        dtype=float
    )

    df_fit["OS"] = np.asarray(
        event_train,
        dtype=int
    )

    try:

        cph = CoxPHFitter(
            penalizer=ELASTIC_NET_PENALIZER,
            l1_ratio=ELASTIC_NET_L1_RATIO,
        )

        cph.fit(
            df_fit,
            duration_col="OS_time",
            event_col="OS",
            show_progress=False,
            fit_options={
                "max_steps": 200
            }
        )

        coef = cph.params_.copy()

        result = pd.DataFrame({
            "Feature": coef.index,
            "Coefficient": coef.values,
        })

        result["Abs_Coefficient"] = (
            result["Coefficient"].abs()
        )

        result["Selected"] = (
            result["Abs_Coefficient"] > 1e-8
        )

        return result.sort_values(
            "Abs_Coefficient",
            ascending=False
        )

    except Exception as e:

        print(
            "ElasticNet selection failed:",
            repr(e)
        )

        return pd.DataFrame(
            columns=[
                "Feature",
                "Coefficient",
                "Abs_Coefficient",
                "Selected"
            ]
        )

In [ ]:
# ============================================================
# Train-only Random Survival Forest
# ============================================================

def train_only_rsf_selection(
    X_train,
    time_train,
    event_train,
    features,
    random_state,
):

    X = X_train[features].copy()

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imp = imputer.fit_transform(X)

    # zero variance
    variances = np.var(
        X_imp,
        axis=0
    )

    keep_mask = variances > 1e-12

    kept_features = [
        f for f, keep
        in zip(features, keep_mask)
        if keep
    ]

    if len(kept_features) == 0:
        return pd.DataFrame(
            columns=[
                "Feature",
                "Importance",
                "Selected"
            ]
        )

    X_imp = X_imp[:, keep_mask]

    y = Surv.from_arrays(
        event=np.asarray(
            event_train,
            dtype=bool
        ),
        time=np.asarray(
            time_train,
            dtype=float
        )
    )

    rsf = RandomSurvivalForest(
        n_estimators=RSF_N_ESTIMATORS,
        min_samples_split=RSF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
        max_features=RSF_MAX_FEATURES,
        random_state=random_state,
        n_jobs=RSF_N_JOBS,
    )

    rsf.fit(
        X_imp,
        y
    )

    perm = permutation_importance(
        rsf,
        X_imp,
        y,
        n_repeats=RSF_PERMUTATION_REPEATS,
        random_state=random_state,
        n_jobs=RSF_N_JOBS,
    )

    result = pd.DataFrame({
        "Feature": kept_features,
        "Importance": perm.importances_mean,
        "Importance_SD": perm.importances_std,
    })

    result = result.sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    result["Selected"] = False

    top_n = min(
        RSF_TOP_N,
        len(result)
    )

    result.loc[
        :top_n - 1,
        "Selected"
    ] = True

    return result

In [ ]:
# ============================================================
# Train-only Multi-method Feature Selection
#
# Univariate Cox
#       +
# ElasticNet Cox
#       +
# RSF
#
# Strict consensus = 3/3
# Majority consensus = >=2/3
# ============================================================

def train_only_multimethod_selection(
    train_df,
    feature_pool,
    random_state,
):

    X_train = train_df[feature_pool]

    time_train = train_df[TIME_COL].values
    event_train = train_df[EVENT_COL].values

    # --------------------------------------------------------
    # Method 1: Univariate Cox
    # --------------------------------------------------------

    uni = train_only_univariate_cox(
        X_train=X_train,
        time_train=time_train,
        event_train=event_train,
        features=feature_pool,
    )

    uni_selected = set(
        uni.loc[
            uni["Selected"],
            "Feature"
        ]
    ) if not uni.empty else set()

    # --------------------------------------------------------
    # Method 2: ElasticNet
    # --------------------------------------------------------

    en = train_only_elastic_net_selection(
        X_train=X_train,
        time_train=time_train,
        event_train=event_train,
        features=feature_pool,
    )

    en_selected = set(
        en.loc[
            en["Selected"],
            "Feature"
        ]
    ) if not en.empty else set()

    # --------------------------------------------------------
    # Method 3: RSF
    # --------------------------------------------------------

    if RUN_RSF:

        rsf = train_only_rsf_selection(
            X_train=X_train,
            time_train=time_train,
            event_train=event_train,
            features=feature_pool,
            random_state=random_state,
        )

        rsf_selected = set(
            rsf.loc[
                rsf["Selected"],
                "Feature"
            ]
        ) if not rsf.empty else set()

    else:

        rsf = pd.DataFrame(
            columns=[
                "Feature",
                "Importance",
                "Selected"
            ]
        )

        rsf_selected = set()

    # --------------------------------------------------------
    # Selection count
    # --------------------------------------------------------

    selection_records = []

    for feature in feature_pool:

        uni_flag = int(
            feature in uni_selected
        )

        en_flag = int(
            feature in en_selected
        )

        rsf_flag = int(
            feature in rsf_selected
        )

        count = (
            uni_flag
            + en_flag
            + rsf_flag
        )

        selection_records.append({
            "Feature": feature,
            "Univariate_Cox": uni_flag,
            "ElasticNet": en_flag,
            "RSF": rsf_flag,
            "Selection_Count": count,
            "Strict_Consensus_3of3": count == 3,
            "Majority_Consensus_2of3": count >= 2,
        })

    consensus = pd.DataFrame(
        selection_records
    )

    return {
        "univariate": uni,
        "elastic_net": en,
        "rsf": rsf,
        "consensus": consensus,
        "strict_features": sorted(
            consensus.loc[
                consensus["Strict_Consensus_3of3"],
                "Feature"
            ].tolist()
        ),
        "majority_features": sorted(
            consensus.loc[
                consensus["Majority_Consensus_2of3"],
                "Feature"
            ].tolist()
        ),
    }

In [ ]:
# ============================================================
# 执行 Gene / Biological 两个 Feature Pool 的稳定性选择
# ============================================================

selection_logs = []
selection_summary = []

for split_info in CV_SPLITS:

    repeat_id = split_info["repeat"]
    fold_id = split_info["fold"]

    train_idx = split_info["train_idx"]

    train_df = analysis_df.iloc[
        train_idx
    ].copy()

    print(
        f"\nRepeat {repeat_id} | Fold {fold_id}"
    )

    # ========================================================
    # Gene pool
    # ========================================================

    gene_selection = train_only_multimethod_selection(
        train_df=train_df,
        feature_pool=GENE_POOL,
        random_state=RANDOM_STATE + repeat_id * 100 + fold_id,
    )

    # ========================================================
    # Biological pool
    # ========================================================

    bio_selection = train_only_multimethod_selection(
        train_df=train_df,
        feature_pool=BIO_POOL,
        random_state=RANDOM_STATE + 1000 + repeat_id * 100 + fold_id,
    )

    for family_name, result in [
        ("Gene", gene_selection),
        ("Biological", bio_selection),
    ]:

        consensus = result["consensus"].copy()

        consensus["Repeat"] = repeat_id
        consensus["Fold"] = fold_id
        consensus["Feature_Family"] = family_name

        selection_logs.append(
            consensus
        )

        selection_summary.append({
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Feature_Family": family_name,
            "N_Strict_3of3": len(
                result["strict_features"]
            ),
            "N_Majority_2of3": len(
                result["majority_features"]
            ),
            "Strict_Features": ";".join(
                result["strict_features"]
            ),
            "Majority_Features": ";".join(
                result["majority_features"]
            ),
        })

    print(
        "  Gene strict:",
        len(gene_selection["strict_features"]),
        "| majority:",
        len(gene_selection["majority_features"])
    )

    print(
        "  Bio strict:",
        len(bio_selection["strict_features"]),
        "| majority:",
        len(bio_selection["majority_features"])
    )

selection_log_df = pd.concat(
    selection_logs,
    ignore_index=True
)

selection_summary_df = pd.DataFrame(
    selection_summary
)

selection_log_df.to_csv(
    RESULTS_DIR / "TrainOnly_Feature_Selection_ByFold.csv",
    index=False,
    encoding="utf-8-sig"
)

selection_summary_df.to_csv(
    RESULTS_DIR / "TrainOnly_Feature_Selection_Summary_ByFold.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTrain-only selection 完成。")

In [ ]:
# ============================================================
# Feature Selection Frequency
# ============================================================

def calculate_selection_frequency(
    selection_log_df,
    family_name,
):

    df = selection_log_df[
        selection_log_df["Feature_Family"] == family_name
    ].copy()

    n_folds = (
        df[["Repeat", "Fold"]]
        .drop_duplicates()
        .shape[0]
    )

    grouped = (
        df.groupby("Feature")
        .agg({
            "Univariate_Cox": "sum",
            "ElasticNet": "sum",
            "RSF": "sum",
            "Selection_Count": "sum",
            "Strict_Consensus_3of3": "sum",
            "Majority_Consensus_2of3": "sum",
        })
        .reset_index()
    )

    grouped["Feature_Family"] = family_name
    grouped["N_Folds"] = n_folds

    grouped["Univariate_Frequency"] = (
        grouped["Univariate_Cox"] / n_folds
    )

    grouped["ElasticNet_Frequency"] = (
        grouped["ElasticNet"] / n_folds
    )

    grouped["RSF_Frequency"] = (
        grouped["RSF"] / n_folds
    )

    grouped["Strict_Consensus_Frequency"] = (
        grouped["Strict_Consensus_3of3"] / n_folds
    )

    grouped["Majority_Consensus_Frequency"] = (
        grouped["Majority_Consensus_2of3"] / n_folds
    )

    grouped["High_Stability_Strict"] = (
        grouped["Strict_Consensus_Frequency"]
        >= STABILITY_FREQ_THRESHOLD
    )

    return grouped.sort_values(
        "Strict_Consensus_Frequency",
        ascending=False
    )


gene_frequency = calculate_selection_frequency(
    selection_log_df,
    "Gene"
)

bio_frequency = calculate_selection_frequency(
    selection_log_df,
    "Biological"
)

frequency_df = pd.concat(
    [
        gene_frequency,
        bio_frequency,
    ],
    ignore_index=True
)

frequency_df.to_csv(
    RESULTS_DIR / "Feature_Selection_Frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    frequency_df.head(30)
)

In [ ]:
# ============================================================
# Step15B Stability Candidates
# ============================================================

stability_candidates = frequency_df[
    frequency_df["High_Stability_Strict"]
].copy()

stability_candidates = stability_candidates.sort_values(
    [
        "Feature_Family",
        "Strict_Consensus_Frequency",
    ],
    ascending=[
        True,
        False,
    ]
)

stability_candidates.to_csv(
    RESULTS_DIR / "Step15B_HighStability_Candidates_Exploratory.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    stability_candidates
)

In [ ]:
# ============================================================
# Survival Model Fit / Predict
# ============================================================

def prepare_features(
    train_df,
    valid_df,
    features,
):

    X_train = train_df[features].copy()
    X_valid = valid_df[features].copy()

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train_imp = imputer.fit_transform(
        X_train
    )

    X_valid_imp = imputer.transform(
        X_valid
    )

    return (
        X_train_imp,
        X_valid_imp,
        imputer,
    )


def fit_predict_cox(
    train_df,
    valid_df,
    features,
):

    X_train, X_valid, _ = prepare_features(
        train_df,
        valid_df,
        features,
    )

    # standardization
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_valid_scaled = scaler.transform(
        X_valid
    )

    train_fit = pd.DataFrame(
        X_train_scaled,
        columns=features,
        index=train_df.index,
    )

    train_fit[TIME_COL] = train_df[TIME_COL].values
    train_fit[EVENT_COL] = train_df[EVENT_COL].values

    cph = CoxPHFitter(
        penalizer=1e-6
    )

    cph.fit(
        train_fit,
        duration_col=TIME_COL,
        event_col=EVENT_COL,
        show_progress=False,
        fit_options={
            "max_steps": 200
        }
    )

    valid_fit = pd.DataFrame(
        X_valid_scaled,
        columns=features,
        index=valid_df.index,
    )

    risk = (
        cph.predict_partial_hazard(
            valid_fit
        )
        .values
        .reshape(-1)
    )

    return risk


def fit_predict_elastic_net(
    train_df,
    valid_df,
    features,
):

    X_train, X_valid, _ = prepare_features(
        train_df,
        valid_df,
        features,
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_valid_scaled = scaler.transform(
        X_valid
    )

    train_fit = pd.DataFrame(
        X_train_scaled,
        columns=features,
        index=train_df.index,
    )

    train_fit[TIME_COL] = train_df[TIME_COL].values
    train_fit[EVENT_COL] = train_df[EVENT_COL].values

    cph = CoxPHFitter(
        penalizer=ELASTIC_NET_PENALIZER,
        l1_ratio=ELASTIC_NET_L1_RATIO,
    )

    cph.fit(
        train_fit,
        duration_col=TIME_COL,
        event_col=EVENT_COL,
        show_progress=False,
        fit_options={
            "max_steps": 200
        }
    )

    valid_fit = pd.DataFrame(
        X_valid_scaled,
        columns=features,
        index=valid_df.index,
    )

    risk = (
        cph.predict_partial_hazard(
            valid_fit
        )
        .values
        .reshape(-1)
    )

    return risk


def fit_predict_rsf(
    train_df,
    valid_df,
    features,
    random_state,
):

    X_train, X_valid, _ = prepare_features(
        train_df,
        valid_df,
        features,
    )

    y_train = Surv.from_arrays(
        event=train_df[EVENT_COL].astype(bool).values,
        time=train_df[TIME_COL].astype(float).values,
    )

    rsf = RandomSurvivalForest(
        n_estimators=RSF_N_ESTIMATORS,
        min_samples_split=RSF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
        max_features=RSF_MAX_FEATURES,
        random_state=random_state,
        n_jobs=RSF_N_JOBS,
    )

    rsf.fit(
        X_train,
        y_train
    )

    risk = rsf.predict(
        X_valid
    )

    return risk

In [ ]:
# ============================================================
# C-index
# ============================================================

def calculate_cindex(
    time,
    event,
    risk,
):

    result = concordance_index_censored(
        event.astype(bool),
        time.astype(float),
        risk.astype(float),
    )

    return float(result[0])

In [ ]:
# ============================================================
# Repeated CV - Fixed Feature Sets
# ============================================================

cv_performance_records = []
oof_prediction_records = []

ALGORITHMS = [
    "CoxPH",
    "ElasticNet_Cox",
]

if RUN_RSF:
    ALGORITHMS.append("RSF")


for split_info in CV_SPLITS:

    repeat_id = split_info["repeat"]
    fold_id = split_info["fold"]

    train_idx = split_info["train_idx"]
    valid_idx = split_info["valid_idx"]

    train_df = analysis_df.iloc[
        train_idx
    ].copy()

    valid_df = analysis_df.iloc[
        valid_idx
    ].copy()

    print(
        f"\nRepeat {repeat_id} | Fold {fold_id}"
    )

    for model_name, features in MODEL_FEATURE_SETS.items():

        print(
            f"  {model_name}"
        )

        # ----------------------------------------------------
        # CoxPH
        # ----------------------------------------------------

        try:

            risk = fit_predict_cox(
                train_df,
                valid_df,
                features,
            )

            cindex = calculate_cindex(
                valid_df[TIME_COL].values,
                valid_df[EVENT_COL].values,
                risk,
            )

            cv_performance_records.append({
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Model": model_name,
                "Algorithm": "CoxPH",
                "N_Features": len(features),
                "C_Index": cindex,
                "N_Train": len(train_df),
                "N_Validation": len(valid_df),
            })

            for sid, r in zip(
                valid_df[master_id_col].astype(str),
                risk,
            ):

                oof_prediction_records.append({
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Sample_ID": sid,
                    "Model": model_name,
                    "Algorithm": "CoxPH",
                    "RiskScore": float(r),
                })

        except Exception as e:

            print(
                "    CoxPH failed:",
                repr(e)
            )

        # ----------------------------------------------------
        # ElasticNet Cox
        # ----------------------------------------------------

        try:

            risk = fit_predict_elastic_net(
                train_df,
                valid_df,
                features,
            )

            cindex = calculate_cindex(
                valid_df[TIME_COL].values,
                valid_df[EVENT_COL].values,
                risk,
            )

            cv_performance_records.append({
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Model": model_name,
                "Algorithm": "ElasticNet_Cox",
                "N_Features": len(features),
                "C_Index": cindex,
                "N_Train": len(train_df),
                "N_Validation": len(valid_df),
            })

            for sid, r in zip(
                valid_df[master_id_col].astype(str),
                risk,
            ):

                oof_prediction_records.append({
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Sample_ID": sid,
                    "Model": model_name,
                    "Algorithm": "ElasticNet_Cox",
                    "RiskScore": float(r),
                })

        except Exception as e:

            print(
                "    ElasticNet failed:",
                repr(e)
            )

        # ----------------------------------------------------
        # RSF
        # ----------------------------------------------------

        if RUN_RSF:

            try:

                risk = fit_predict_rsf(
                    train_df,
                    valid_df,
                    features,
                    random_state=(
                        RANDOM_STATE
                        + repeat_id * 100
                        + fold_id
                    ),
                )

                cindex = calculate_cindex(
                    valid_df[TIME_COL].values,
                    valid_df[EVENT_COL].values,
                    risk,
                )

                cv_performance_records.append({
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Model": model_name,
                    "Algorithm": "RSF",
                    "N_Features": len(features),
                    "C_Index": cindex,
                    "N_Train": len(train_df),
                    "N_Validation": len(valid_df),
                })

                for sid, r in zip(
                    valid_df[master_id_col].astype(str),
                    risk,
                ):

                    oof_prediction_records.append({
                        "Repeat": repeat_id,
                        "Fold": fold_id,
                        "Sample_ID": sid,
                        "Model": model_name,
                        "Algorithm": "RSF",
                        "RiskScore": float(r),
                    })

            except Exception as e:

                print(
                    "    RSF failed:",
                    repr(e)
                )


cv_performance_df = pd.DataFrame(
    cv_performance_records
)

oof_predictions_df = pd.DataFrame(
    oof_prediction_records
)

cv_performance_df.to_csv(
    RESULTS_DIR / "RepeatedCV_Fold_Performance.csv",
    index=False,
    encoding="utf-8-sig"
)

oof_predictions_df.to_csv(
    RESULTS_DIR / "RepeatedCV_OOF_Predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nRepeated CV 完成。")

In [ ]:
# ============================================================
# Repeated CV Performance Summary
# ============================================================

cv_summary = (
    cv_performance_df
    .groupby(
        ["Model", "Algorithm"]
    )["C_Index"]
    .agg(
        Mean="mean",
        SD="std",
        Median="median",
        Min="min",
        Max="max",
        N="count",
    )
    .reset_index()
)

cv_summary["Mean_SD"] = (
    cv_summary["Mean"].round(4).astype(str)
    + " ± "
    + cv_summary["SD"].round(4).astype(str)
)

cv_summary.to_csv(
    RESULTS_DIR / "RepeatedCV_Performance_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    cv_summary.sort_values(
        "Mean",
        ascending=False
    )
)

In [ ]:
# ============================================================
# Mean OOF prediction across 5 repeats
# ============================================================

mean_oof = (
    oof_predictions_df
    .groupby(
        [
            "Sample_ID",
            "Model",
            "Algorithm",
        ]
    )["RiskScore"]
    .mean()
    .reset_index()
)

mean_oof.to_csv(
    RESULTS_DIR / "RepeatedCV_Mean_OOF_RiskScores.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    mean_oof.head()
)

In [ ]:
# ============================================================
# Bootstrap C-index 95% CI
# ============================================================

def stratified_bootstrap_indices(
    event,
    rng,
):

    event = np.asarray(event).astype(int)

    event_idx = np.where(event == 1)[0]
    censor_idx = np.where(event == 0)[0]

    if len(event_idx) == 0 or len(censor_idx) == 0:
        return rng.integers(
            0,
            len(event),
            size=len(event)
        )

    boot_event = rng.choice(
        event_idx,
        size=len(event_idx),
        replace=True
    )

    boot_censor = rng.choice(
        censor_idx,
        size=len(censor_idx),
        replace=True
    )

    return np.concatenate([
        boot_event,
        boot_censor,
    ])


bootstrap_records = []

sample_id_to_row = {
    str(sid): i
    for i, sid in enumerate(
        analysis_df[master_id_col].astype(str)
    )
}

time_all = analysis_df[TIME_COL].values
event_all = analysis_df[EVENT_COL].values

for (model_name, algorithm), group in mean_oof.groupby(
    ["Model", "Algorithm"]
):

    prediction_map = dict(
        zip(
            group["Sample_ID"].astype(str),
            group["RiskScore"]
        )
    )

    common_ids = [
        sid
        for sid in sample_id_to_row
        if sid in prediction_map
    ]

    indices = [
        sample_id_to_row[sid]
        for sid in common_ids
    ]

    times = time_all[indices]
    events = event_all[indices]

    risks = np.array([
        prediction_map[sid]
        for sid in common_ids
    ])

    point_cindex = calculate_cindex(
        times,
        events,
        risks
    )

    rng = np.random.default_rng(
        RANDOM_STATE
        + int.from_bytes(hashlib.sha256((model_name + algorithm).encode("utf-8")).digest()[:4], "little") % 100000
    )

    boot_values = []

    for b in range(N_BOOTSTRAP):

        idx = stratified_bootstrap_indices(
            events,
            rng
        )

        try:

            c = calculate_cindex(
                times[idx],
                events[idx],
                risks[idx],
            )

            boot_values.append(c)

        except Exception:
            continue

    boot_values = np.asarray(
        boot_values
    )

    bootstrap_records.append({
        "Model": model_name,
        "Algorithm": algorithm,
        "Point_C_Index": point_cindex,
        "Bootstrap_N": len(boot_values),
        "Bootstrap_Mean": np.mean(boot_values),
        "CI_2.5": np.percentile(
            boot_values,
            2.5
        ),
        "CI_50": np.percentile(
            boot_values,
            50
        ),
        "CI_97.5": np.percentile(
            boot_values,
            97.5
        ),
    })

bootstrap_df = pd.DataFrame(
    bootstrap_records
)

bootstrap_df.to_csv(
    RESULTS_DIR / "RepeatedCV_Bootstrap_CIndex_95CI.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    bootstrap_df.sort_values(
        "Point_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# Dynamic Train-only Selection Performance
# ============================================================

dynamic_records = []
dynamic_predictions = []

if RUN_DYNAMIC_SELECTION_PERFORMANCE:

    for split_info in CV_SPLITS:

        repeat_id = split_info["repeat"]
        fold_id = split_info["fold"]

        train_idx = split_info["train_idx"]
        valid_idx = split_info["valid_idx"]

        train_df = analysis_df.iloc[
            train_idx
        ].copy()

        valid_df = analysis_df.iloc[
            valid_idx
        ].copy()

        print(
            f"\nDynamic selection:"
            f" Repeat {repeat_id} | Fold {fold_id}"
        )

        # ----------------------------------------------------
        # Gene
        # ----------------------------------------------------

        gene_sel = train_only_multimethod_selection(
            train_df,
            GENE_POOL,
            random_state=(
                RANDOM_STATE
                + 5000
                + repeat_id * 100
                + fold_id
            )
        )

        # ----------------------------------------------------
        # Biological
        # ----------------------------------------------------

        bio_sel = train_only_multimethod_selection(
            train_df,
            BIO_POOL,
            random_state=(
                RANDOM_STATE
                + 10000
                + repeat_id * 100
                + fold_id
            )
        )

        dynamic_feature_sets = {

            "Dynamic_Gene_3of3":
                gene_sel["strict_features"],

            "Dynamic_Bio_3of3":
                bio_sel["strict_features"],

            "Dynamic_Integrated_3of3":
                list(
                    dict.fromkeys(
                        gene_sel["strict_features"]
                        + bio_sel["strict_features"]
                    )
                ),
        }

        for model_name, features in dynamic_feature_sets.items():

            print(
                f"  {model_name}:"
                f" N={len(features)}"
            )

            if len(features) == 0:

                dynamic_records.append({
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Model": model_name,
                    "Algorithm": "CoxPH",
                    "N_Features": 0,
                    "C_Index": np.nan,
                })

                continue

            # ------------------------------------------------
            # CoxPH
            # ------------------------------------------------

            try:

                risk = fit_predict_cox(
                    train_df,
                    valid_df,
                    features,
                )

                cindex = calculate_cindex(
                    valid_df[TIME_COL].values,
                    valid_df[EVENT_COL].values,
                    risk,
                )

                dynamic_records.append({
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Model": model_name,
                    "Algorithm": "CoxPH",
                    "N_Features": len(features),
                    "C_Index": cindex,
                })

                for sid, r in zip(
                    valid_df[master_id_col].astype(str),
                    risk,
                ):

                    dynamic_predictions.append({
                        "Repeat": repeat_id,
                        "Fold": fold_id,
                        "Sample_ID": sid,
                        "Model": model_name,
                        "Algorithm": "CoxPH",
                        "RiskScore": float(r),
                    })

            except Exception as e:

                print(
                    "    Dynamic CoxPH failed:",
                    repr(e)
                )

            # ------------------------------------------------
            # RSF
            # ------------------------------------------------

            if RUN_RSF:

                try:

                    risk = fit_predict_rsf(
                        train_df,
                        valid_df,
                        features,
                        random_state=(
                            RANDOM_STATE
                            + repeat_id * 100
                            + fold_id
                        ),
                    )

                    cindex = calculate_cindex(
                        valid_df[TIME_COL].values,
                        valid_df[EVENT_COL].values,
                        risk,
                    )

                    dynamic_records.append({
                        "Repeat": repeat_id,
                        "Fold": fold_id,
                        "Model": model_name,
                        "Algorithm": "RSF",
                        "N_Features": len(features),
                        "C_Index": cindex,
                    })

                    for sid, r in zip(
                        valid_df[master_id_col].astype(str),
                        risk,
                    ):

                        dynamic_predictions.append({
                            "Repeat": repeat_id,
                            "Fold": fold_id,
                            "Sample_ID": sid,
                            "Model": model_name,
                            "Algorithm": "RSF",
                            "RiskScore": float(r),
                        })

                except Exception as e:

                    print(
                        "    Dynamic RSF failed:",
                        repr(e)
                    )

    dynamic_performance_df = pd.DataFrame(
        dynamic_records
    )

    dynamic_predictions_df = pd.DataFrame(
        dynamic_predictions
    )

    dynamic_performance_df.to_csv(
        RESULTS_DIR / "Dynamic_TrainOnly_Selection_Performance.csv",
        index=False,
        encoding="utf-8-sig"
    )

    dynamic_predictions_df.to_csv(
        RESULTS_DIR / "Dynamic_TrainOnly_OOF_Predictions.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nDynamic train-only model 完成。")

In [ ]:
# ============================================================
# Dynamic Model Summary
# ============================================================

if RUN_DYNAMIC_SELECTION_PERFORMANCE:

    dynamic_summary = (
        dynamic_performance_df
        .groupby(
            ["Model", "Algorithm"]
        )["C_Index"]
        .agg(
            Mean="mean",
            SD="std",
            Median="median",
            Min="min",
            Max="max",
            N="count",
        )
        .reset_index()
    )

    dynamic_feature_summary = (
        dynamic_performance_df
        .groupby(
            ["Model"]
        )["N_Features"]
        .agg(
            Mean_N_Features="mean",
            Median_N_Features="median",
            Min_N_Features="min",
            Max_N_Features="max",
        )
        .reset_index()
    )

    dynamic_summary = dynamic_summary.merge(
        dynamic_feature_summary,
        on="Model",
        how="left"
    )

    dynamic_summary.to_csv(
        RESULTS_DIR / "Dynamic_TrainOnly_Model_Summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(
        dynamic_summary
    )

In [ ]:
# ============================================================
# Figure 1
# Repeated CV C-index distribution
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 7)
)

plot_df = cv_performance_df.copy()

models_for_plot = [
    "StableGenes10",
    "StableBio7",
    "Genes10_Bio7",
    "Clinical",
    "Clinical_RiskScore",
]

plot_df = plot_df[
    plot_df["Model"].isin(
        models_for_plot
    )
]

plot_df.boxplot(
    column="C_Index",
    by=["Model", "Algorithm"],
    ax=ax,
    rot=45
)

ax.set_title(
    "Repeated 5×5 CV C-index Distribution"
)

ax.set_ylabel(
    "C-index"
)

ax.set_xlabel(
    ""
)

plt.suptitle("")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_RepeatedCV_CIndex_Boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_RepeatedCV_CIndex_Boxplot.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Figure 2
# Gene selection stability
# ============================================================

gene_plot = (
    gene_frequency
    .sort_values(
        "Strict_Consensus_Frequency",
        ascending=False
    )
    .head(20)
    .sort_values(
        "Strict_Consensus_Frequency"
    )
)

fig, ax = plt.subplots(
    figsize=(9, 8)
)

ax.barh(
    gene_plot["Feature"],
    gene_plot["Strict_Consensus_Frequency"]
)

ax.axvline(
    STABILITY_FREQ_THRESHOLD,
    linestyle="--",
)

ax.set_xlabel(
    "Strict consensus selection frequency"
)

ax.set_title(
    "Gene Selection Stability Across Repeated CV"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_Gene_Selection_Stability.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_Gene_Selection_Stability.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Figure 3
# Biological feature stability
# ============================================================

bio_plot = (
    bio_frequency
    .sort_values(
        "Strict_Consensus_Frequency",
        ascending=False
    )
    .head(25)
    .sort_values(
        "Strict_Consensus_Frequency"
    )
)

fig, ax = plt.subplots(
    figsize=(10, 10)
)

ax.barh(
    bio_plot["Feature"],
    bio_plot["Strict_Consensus_Frequency"]
)

ax.axvline(
    STABILITY_FREQ_THRESHOLD,
    linestyle="--",
)

ax.set_xlabel(
    "Strict consensus selection frequency"
)

ax.set_title(
    "Biological Feature Selection Stability"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_Biological_Selection_Stability.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_Biological_Selection_Stability.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Figure 4
# Bootstrap C-index 95% CI
# ============================================================

plot_boot = bootstrap_df.copy()

plot_boot = plot_boot[
    plot_boot["Model"].isin(
        [
            "StableGenes10",
            "StableBio7",
            "Genes10_Bio7",
            "Clinical",
            "Clinical_RiskScore",
        ]
    )
].copy()

plot_boot["Label"] = (
    plot_boot["Model"]
    + " | "
    + plot_boot["Algorithm"]
)

plot_boot = plot_boot.sort_values(
    "Point_C_Index"
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

y = np.arange(
    len(plot_boot)
)

x = plot_boot["Point_C_Index"].values

xerr_low = (
    x
    - plot_boot["CI_2.5"].values
)

xerr_high = (
    plot_boot["CI_97.5"].values
    - x
)

ax.errorbar(
    x,
    y,
    xerr=[
        xerr_low,
        xerr_high,
    ],
    fmt="o",
    capsize=4,
)

ax.set_yticks(y)

ax.set_yticklabels(
    plot_boot["Label"]
)

ax.set_xlabel(
    "C-index"
)

ax.set_title(
    "Repeated CV + Bootstrap C-index 95% CI"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_Bootstrap_CIndex_95CI.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES_DIR / "Figure_Step15B_Bootstrap_CIndex_95CI.svg",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step15B Final Performance Table
# ============================================================

final_performance = cv_summary.merge(
    bootstrap_df[
        [
            "Model",
            "Algorithm",
            "Point_C_Index",
            "CI_2.5",
            "CI_97.5",
        ]
    ],
    on=["Model", "Algorithm"],
    how="left"
)

final_performance.to_csv(
    RESULTS_DIR / "STEP15B_FINAL_MODEL_PERFORMANCE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    final_performance.sort_values(
        "Point_C_Index",
        ascending=False
    )
)

In [ ]:
# ============================================================
# STEP15B FINAL AUDIT
# ============================================================

audit_lines = []

audit_lines.append("=" * 80)
audit_lines.append("TCGA-LIHC Step15B Stability Validation Audit")
audit_lines.append("=" * 80)
audit_lines.append("")
audit_lines.append(f"Version: {VERSION}")
audit_lines.append(f"Run time: {RUN_TIME}")
audit_lines.append("")
audit_lines.append("ANALYSIS COHORT")
audit_lines.append("-" * 80)
audit_lines.append(
    f"Master feature table shape: {master.shape}"
)
audit_lines.append(
    f"Step15B analysis cohort: {analysis_df.shape}"
)
audit_lines.append(
    f"OS events: {analysis_df[EVENT_COL].sum()}"
)
audit_lines.append("")
audit_lines.append("FEATURE DEFINITIONS")
audit_lines.append("-" * 80)
audit_lines.append(
    f"Original Top10: {len(ORIGINAL_TOP10)}"
)
audit_lines.append(
    f"Hub18: {len(HUB18)}"
)
audit_lines.append(
    f"Gene candidate pool: {len(GENE_POOL)}"
)
audit_lines.append(
    f"Biological candidate pool: {len(BIO_POOL)}"
)
audit_lines.append(
    f"StableGenes10: {len(STABLE_GENES10)}"
)
audit_lines.append(
    f"StableBio7: {len(STABLE_BIO7)}"
)
audit_lines.append(
    f"Genes10_Bio7: {len(GENES10_BIO7)}"
)
audit_lines.append("")
audit_lines.append("CROSS VALIDATION")
audit_lines.append("-" * 80)
audit_lines.append(
    f"Repeated CV: {N_REPEATS} × {N_SPLITS}"
)
audit_lines.append(
    f"Total outer folds: {len(CV_SPLITS)}"
)
audit_lines.append(
    "Feature selection: train-only"
)
audit_lines.append(
    "Validation folds were not used for feature selection."
)
audit_lines.append("")
audit_lines.append("BOOTSTRAP")
audit_lines.append("-" * 80)
audit_lines.append(
    f"Bootstrap iterations: {N_BOOTSTRAP}"
)
audit_lines.append(
    "Bootstrap performed on repeated-CV mean OOF predictions."
)
audit_lines.append("")
audit_lines.append("IMPORTANT METHODOLOGICAL NOTE")
audit_lines.append("-" * 80)
audit_lines.append(
    "Step15B evaluates internal stability and repeated internal validation."
)
audit_lines.append(
    "It is not an independent external validation."
)
audit_lines.append(
    "RiskScore_Historical_Benchmark is retained as a historical benchmark "
    "and should not be interpreted as a leakage-free de novo model."
)
audit_lines.append(
    "StableGenes10 and StableBio7 remain locked Step15A definitions; "
    "Step15B does not overwrite them."
)
audit_lines.append(
    "Fold-specific dynamic selection is evaluated separately from "
    "the locked feature-set benchmark."
)
audit_lines.append("")
audit_lines.append("OUTPUTS")
audit_lines.append("-" * 80)

for p in sorted(RESULTS_DIR.glob("*")):
    if p.is_file():
        audit_lines.append(str(p))

audit_lines.append("")
audit_lines.append("STEP15B STATUS: COMPLETE")
audit_lines.append("=" * 80)

audit_text = "\n".join(audit_lines)

with open(
    AUDIT_DIR / "STEP15B_STABILITY_VALIDATION_AUDIT.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(audit_text)

print(audit_text)

In [ ]:
# ============================================================
# Step15B Summary JSON
# ============================================================

summary_json = {
    "Project": "TCGA-LIHC",
    "Step": "Step15B",
    "Version": VERSION,

    "Analysis_N": int(len(analysis_df)),
    "Events": int(
        analysis_df[EVENT_COL].sum()
    ),

    "Original_Top10_N": len(ORIGINAL_TOP10),
    "Hub18_N": len(HUB18),
    "Gene_Pool_N": len(GENE_POOL),
    "Biological_Pool_N": len(BIO_POOL),

    "StableGenes10_N": len(STABLE_GENES10),
    "StableBio7_N": len(STABLE_BIO7),
    "Genes10_Bio7_N": len(GENES10_BIO7),

    "RepeatedCV": f"{N_REPEATS}x{N_SPLITS}",
    "Outer_Folds": len(CV_SPLITS),

    "Bootstrap_N": N_BOOTSTRAP,

    "Dynamic_TrainOnly_Selection": (
        RUN_DYNAMIC_SELECTION_PERFORMANCE
    ),

    "RSF": RUN_RSF,

    "Output_Directory": str(RESULTS_DIR),
    "Figure_Directory": str(FIGURES_DIR),
    "Audit_Directory": str(AUDIT_DIR),
}

with open(
    AUDIT_DIR / "STEP15B_FINAL_SUMMARY.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    json.dumps(
        summary_json,
        ensure_ascii=False,
        indent=2
    )
)